# 03 — R-Peak Detection (Pan-Tompkins-style)

Building the pipeline stage by stage: derivative -> squaring -> moving-window integration -> peak detection. This notebook starts with just the derivative stage.

## Load and filter the record

Same record and filter as `02_signal_processing.ipynb`.

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

RECORD_NAME = "100"

record = wfdb.rdrecord(RECORD_NAME, pn_dir="mitdb")
fs = record.fs
lead_index = record.sig_name.index("MLII") if "MLII" in record.sig_name else 0
ecg_signal = record.p_signal[:, lead_index]
time = np.arange(len(ecg_signal)) / fs


def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    """Zero-phase Butterworth bandpass filter for ECG."""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)


filtered_signal = bandpass_filter(ecg_signal, lowcut=0.5, highcut=40, fs=fs, order=4)

## Stage 1: Derivative

Approximates the local slope (rate of change) of the signal at each sample.
We use `np.gradient`, which computes a central difference (using the
neighboring sample on each side) and divides by the true sample spacing
`1/fs`, giving output in physically meaningful units (mV/s) rather than raw
per-sample differences. This is a simplified stand-in for the weighted
5-point derivative used in the original Pan-Tompkins paper — good enough
for our purposes, and easy to swap out later if needed.

In [ ]:
def derivative_filter(signal, fs):
    """Approximate slope (dV/dt) of the signal using a central difference."""
    return np.gradient(signal, 1 / fs)


derivative_signal = derivative_filter(filtered_signal, fs)

## Filtered ECG vs its derivative

**What to look for:** the P and T waves, which change gently, should almost
flatten out in the derivative. The QRS complex, which rises and falls in a
few tens of milliseconds, should produce a sharp positive-then-negative
spike pair — and nothing else in the signal should come close to that
magnitude.

In [ ]:
zoom_start, zoom_end = 60, 63  # seconds
mask = (time >= zoom_start) & (time < zoom_end)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(time[mask], filtered_signal[mask], color="tab:blue", linewidth=1.0)
axes[0].set_title("Filtered ECG")
axes[0].set_ylabel("Amplitude (mV)")

axes[1].plot(time[mask], derivative_signal[mask], color="tab:orange", linewidth=1.0)
axes[1].set_title("Derivative of filtered ECG")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("dV/dt (mV/s)")

fig.tight_layout()
fig.savefig("../results/figures/04_derivative.png", dpi=120)
plt.show()